# Kintsugi: Reproducible Demo and Diagnostic Report

This notebook reproduces the complete Kintsugi pipeline on a synthetic
dataset, generates the diagnostic report, and visualises the tessellation
output.  It is self-contained: no external data download is needed.

**Requirements**: `pip install kintsugi-st` (or `pip install -e ".[dev]"` from source).

In [ ]:
import hashlib
import time

import numpy as np
import kintsugi

print(f"kintsugi {kintsugi.__version__}")

## 1. Generate synthetic dataset

The toy dataset is a 60x60 grid with four spatially distinct domains
(different density and marker-gene programmes), masked to a circular
tissue region.  It is bundled inside the package and requires no download.

In [ ]:
grid = kintsugi.make_toy_dataset()
print(f"Grid:        {grid.rows} x {grid.cols}")
print(f"Genes:       {grid.n_genes}")
print(f"Tissue bins: {int(grid.mask.sum()):,}")
print(f"Total UMI:   {int(np.asarray(grid.counts.sum())):,}")

## 2. Run tessellation

In [ ]:
t0 = time.perf_counter()
result = grid.tessellate()
elapsed = time.perf_counter() - t0

print(f"Regions: {result.n_regions}")
print(f"Time:    {elapsed:.2f} s")

# Determinism check
label_hash = hashlib.sha256(result.labels.tobytes()).hexdigest()[:16]
print(f"Label hash (sha256[:16]): {label_hash}")
assert label_hash == "3653373eec97b137", f"Hash mismatch: {label_hash}"
print("Determinism check PASSED")

## 3. Diagnostic report

The report computes 8 metrics: region count, median UMI, density CV,
Poisson stationarity pass rate, held-out log-likelihoods for composition
and density, improvement over a global null, and a composition-dominates
warning.

In [ ]:
report = kintsugi.tessellation_report(result, grid)
print(report)

## 4. Visualise tessellation

Plot the boundary-tensor trace (left), region labels (centre), and
per-region UMI density (right).  Requires matplotlib (not a core
dependency).

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    raise ImportError("pip install matplotlib  # required for visualisation cells")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Trace field
trace_masked = np.where(grid.mask, result.trace, np.nan)
im0 = axes[0].imshow(trace_masked, cmap="hot", interpolation="nearest")
axes[0].set_title("Boundary-tensor trace")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# Region labels
labels_display = np.where(grid.mask, result.labels, np.nan)
im1 = axes[1].imshow(labels_display, cmap="nipy_spectral", interpolation="nearest")
axes[1].set_title(f"Region labels (K={result.n_regions})")

# Per-region UMI density
density_map = np.full_like(result.trace, np.nan)
for k in range(result.n_regions):
    region_mask = result.labels == k
    density_map[region_mask] = result.depths[k] / max(result.areas[k], 1)
im2 = axes[2].imshow(density_map, cmap="viridis", interpolation="nearest")
axes[2].set_title("UMI density per region")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

## 5. AnnData export (scverse integration)

In [ ]:
try:
    adata = kintsugi.to_anndata(result, grid=grid, use_raw_counts=True)
    print(f"Shape:  {adata.shape}")
    print(f"obs:    {list(adata.obs.columns)}")
    print(f"obsm:   {list(adata.obsm.keys())}")
    print(f"obsp:   {list(adata.obsp.keys())}")
    print(f"layers: {list(adata.layers.keys())}")
    print(f"\nadata.obs.head():")
    print(adata.obs.head())
except ImportError:
    print("anndata not installed. Install with: pip install 'kintsugi-st[anndata]'")

## 6. Region-size and depth distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].hist(result.areas, bins=30, edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Bins per region")
axes[0].set_ylabel("Count")
axes[0].set_title("Region area distribution")
axes[0].axvline(np.median(result.areas), color="red", ls="--",
                label=f"median = {np.median(result.areas):.0f}")
axes[0].legend()

axes[1].hist(result.depths, bins=30, edgecolor="white", linewidth=0.5, color="C1")
axes[1].set_xlabel("Total UMI per region")
axes[1].set_ylabel("Count")
axes[1].set_title("Region depth distribution")
axes[1].axvline(np.median(result.depths), color="red", ls="--",
                label=f"median = {np.median(result.depths):.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Runtime and memory benchmark

Benchmarks on synthetic grids of increasing size.  All timings are
single-threaded, pure Python, on the current machine.

In [ ]:
import tracemalloc
import scipy.sparse as sp

configs = [
    ("Toy",    60,  60,  100),
    ("Small", 100, 100,  200),
    ("Medium",200, 200,  500),
    ("Large", 500, 500, 1000),
]

print(f"{'Dataset':<10s}  {'Grid':>9s}  {'Genes':>5s}  {'Tissue':>8s}  "
      f"{'Regions':>7s}  {'Time':>7s}  {'Memory':>8s}")
print("-" * 72)

for label, rows, cols, n_genes in configs:
    rng = np.random.default_rng(2024)
    N = rows * cols
    rr, cc = np.mgrid[:rows, :cols]
    cr, cc2 = rows / 2 - 0.5, cols / 2 - 0.5
    mask = np.sqrt((rr - cr)**2 + (cc - cc2)**2) <= min(rows, cols) / 2 - 1
    tissue_idx = np.where(mask.ravel())[0]
    n_tissue = tissue_idx.size
    umi = rng.poisson(8.0, size=n_tissue)
    dr, dc, dv = [], [], []
    for i, fi in enumerate(tissue_idx):
        if umi[i] == 0: continue
        gs = rng.choice(n_genes, min(umi[i], n_genes), replace=False)
        cs = rng.multinomial(umi[i], np.ones(gs.size)/gs.size)
        nz = cs > 0
        for g, v in zip(gs[nz], cs[nz]):
            dr.append(fi); dc.append(g); dv.append(float(v))
    counts = sp.coo_matrix((dv, (dr, dc)), shape=(N, n_genes)).tocsr()
    g = kintsugi.GridData(counts, rows=rows, cols=cols, mask=mask)

    # Warmup
    g.tessellate()

    tracemalloc.start()
    t0 = time.perf_counter()
    res = g.tessellate()
    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    print(f"{label:<10s}  {rows:>3d} x {cols:<3d}  {n_genes:>5d}  "
          f"{n_tissue:>8,d}  {res.n_regions:>7,d}  {elapsed:>6.2f} s  "
          f"{peak/1024/1024:>7.1f} MB")